In [0]:
from pyspark.sql import functions as F

bronze_table = "bronze_telemetry"
silver_table = "silver_telemetry"
silver_checkpoint = "/Volumes/workspace/default/thermal_data/_checkpoints/silver/"

In [0]:
# 1. Read stream from Delta Bronze table
bronze_stream = spark.readStream.table(bronze_table)

# 2. Transformations and Feature Engineering
silver_df = (
    bronze_stream
    # Data Quality / Sanity Filtering
    .filter(
        F.col("timestamp").isNotNull() &
        F.col("cpu_utilization").between(0.0, 100.0) &
        F.col("gpu_utilization").between(0.0, 100.0) &
        (F.col("power_draw_watts") >= 0.0) &
        F.col("current_fluid_temp").between(10.0, 120.0)
    )
    # Feature Engineering
    .withColumn("total_compute_utilization", F.round((F.col("cpu_utilization") + F.col("gpu_utilization")) / 2, 2))
    .withColumn("power_per_compute_unit", F.round(F.col("power_draw_watts") / F.when(F.col("total_compute_utilization") == 0, 1).otherwise(F.col("total_compute_utilization")), 3))
    .withColumn("is_thermal_alert", F.when((F.col("current_fluid_temp") >= 42.0) & (F.col("power_draw_watts") > 300.0), True).otherwise(False))
    .withColumn("transformed_at", F.current_timestamp())
)

# 3. Write Stream to Silver Table
silver_query = (
    silver_df.writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", silver_checkpoint)
    .trigger(availableNow=True)
    .toTable(silver_table)
)

silver_query.awaitTermination()

In [0]:
display(spark.table(silver_table))

In [0]:
display(
    spark.table(silver_table)
    .select(
        "timestamp", 
        "total_compute_utilization", 
        "power_draw_watts", 
        "current_fluid_temp", 
        "is_thermal_alert"
    )
)